### Messing with encodecs

In [1]:
!pip install encodec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 26.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45759 sha256=8d97e67c10124702d358709e1083c7abefbe1f6ec11665fd0f66d06c8189fe69
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec


In [2]:
!pip install soundfile

Two audio files that should be fairly different.

We'll try to extract and apply the tone from one to the other later using questionable latent space shenanigans.

In [130]:
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio
import librosa

device = "cuda" if torch.cuda.is_available() else "cpu"
model = EncodecModel.encodec_model_24khz().to(device)



#auduos
sample1 = '/content/sample1.wav'
sample2 = '/content/noise_sample.wav'


#shift sample 2 down
sample2Audio, modsr = librosa.load(sample2, sr=None)

shifted = librosa.effects.pitch_shift(sample2Audio, sr=modsr, n_steps=-8)

sf.write("sample2Mod.wav", shifted, modsr)

sample2Mod = "sample2Mod.wav"


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Check em

In [14]:
Audio(sample1)

In [15]:
Audio(sample2)

In [18]:
Audio(sample2Mod)

def encoder and decoder

In [158]:
from encodec.model import EncodecModel

def encode(path):
  wav, sr = sf.read(path)

  if wav.ndim > 1:
    wav = wav[:, 0] #single channel

  wav = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

  with torch.no_grad():
    encoded = model.encode(wav)

  codes, _ = encoded[0]     # extract tuple from list

  # Decode integer code ids
  latents = model.quantizer.decode(codes)
  return latents

In [121]:
#!!!!!! probably doesn't work because of changes to encoder side to get latent stuff to work
def decode(latents, path='/content/decoded.wav'):
  # Decode
  with torch.no_grad():
    decoded_audio = model.decode(latents)

  decoded_audio_path = path

  decoded_audio_np = decoded_audio.squeeze(0).squeeze(0).cpu().numpy()

  sf.write(decoded_audio_path, decoded_audio_np, 24000)

Encode

In [159]:
encoded1 = encode(sample1)
encoded2 = encode(sample2)
encoded2mod = encode(sample2Mod)

Abuse in latent space - subtract pitch shifted versions to try and get rid of words or something...

In [160]:

min_len = min(encoded2.shape[-1], encoded2mod.shape[-1])
encoded2 = encoded2[..., :min_len]
encoded2mod = encoded2mod[..., :min_len]

min_batch = min(encoded2.shape[0], encoded2mod.shape[0])
encoded2 = encoded2[:min_batch]
encoded2mod = encoded2mod[:min_batch]

latent_diff = encoded2 - encoded2mod

Clamp and squeeze until it runs

In [166]:
with torch.no_grad():
    audio = model.decoder(latent_diff[0].unsqueeze(0)).clamp(-1, 1) #decoder is different from decode

sf.write("pureSponge.wav", audio.squeeze(0).squeeze(0).cpu().numpy(), 24000)

Pure voice extract.

This sounds good!

In [167]:

Audio('/content/pureSponge.wav')

Do similar process to add to sample 1

In [169]:
min_len = min(encoded1.shape[-1], latent_diff.shape[-1])
encoded1 = encoded1[..., :min_len]
latent_diff = latent_diff[..., :min_len]

min_batch = min(encoded1.shape[0], latent_diff.shape[0])
encoded1 = encoded1[:min_batch]
latent_diff = latent_diff[:min_batch]

latent_add = encoded1 + latent_diff

In [170]:
with torch.no_grad():
    audio = model.decoder(latent_add[0].unsqueeze(0)).clamp(-1, 1)

sf.write("newDragon.wav", audio.squeeze(0).squeeze(0).cpu().numpy(), 24000)

Well, we lost all the words but still have the deep rumble. It added all the pain and none of the sponge.

In [171]:
Audio('/content/newDragon.wav')